In [ ]:
############################################
# SEPARAÇÃO SEMÂNTICA POR CÓDIGO
############################################
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Carregar os dados com as colunas corrigidas
# Mapeamento padronizado das colunas do arquivo Excel

# Coluna A (SN) = identificador único do documento
# Coluna F (BOX) = número de identificação do box físico
# Coluna I (COD) = código de classificação conforme a Tabela de Temporalidade
# Coluna O (DESC) = Descrição do documento

# O Python começa no índice zero
colunas_alvo = [0, 5, 8, 14]
df = pd.read_excel('/content/dados_AP.xlsx', usecols=colunas_alvo, names=['SN', 'BOX','COD', 'DESC'])

# Limpar linhas sem descrição
df = df.dropna(subset=['DESC'])

# 2. Isolar o código
codigo_alvo = "2.0.04.02.01"
df_filtrado = df[df['COD'] == codigo_alvo].copy()

if len(df_filtrado) < 10:
    print("Amostra muito pequena para justificar ramificação.")
    exit()

# 3. Vetorização
stop_words_pt = ['de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'para', 'com', 'os', 'as']
vectorizer = TfidfVectorizer(stop_words=stop_words_pt, max_df=0.85)
X = vectorizer.fit_transform(df_filtrado['DESC'])

# 4. Clusterização
num_clusters = 2
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_filtrado['Novo_Grupo'] = kmeans.fit_predict(X)

# 5. Avaliação Matemática
score = silhouette_score(X, kmeans.labels_)
print(f"--- ANÁLISE DO CÓDIGO: {codigo_alvo} ---")
print(f"Score de Silhueta: {score:.3f}")

# 6. Visualização Gráfica (Redução para 2D com PCA)
# Transforma os vetores de alta dimensão em coordenadas X e Y para o gráfico
pca = PCA(n_components=2, random_state=42)
coordenadas_2d = pca.fit_transform(X.toarray())

plt.figure(figsize=(10, 6))
# Renderiza o gráfico utilizando tons de azul
scatter = plt.scatter(coordenadas_2d[:, 0], coordenadas_2d[:, 1],
                      c=df_filtrado['Novo_Grupo'], cmap='Blues',
                      edgecolor='navy', alpha=0.7, s=100)

plt.title(f"Separação Semântica do Código: {codigo_alvo}", fontsize=14, color='navy')
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 7. Visualizar e Exportar as Descrições Separadas

# Filtra o DataFrame para separar os dois grupos criados pelo algoritmo
grupo_0 = df_filtrado[df_filtrado['Novo_Grupo'] == 0]
grupo_1 = df_filtrado[df_filtrado['Novo_Grupo'] == 1]

print("\n--- AMOSTRAS DO GRUPO 0 ---")
# Mostra as 5 primeiras descrições deste grupo
for texto in grupo_0['DESC'].head(5):
    print(f"- {texto}")

print("\n--- AMOSTRAS DO GRUPO 1 ---")
# Mostra as 5 primeiras descrições deste grupo
for texto in grupo_1['DESC'].head(5):
    print(f"- {texto}")

# 8. Gerar arquivo Excel com o resultado
nome_arquivo_saida = f"analise_divisao_{codigo_alvo.replace('.', '_')}.xlsx"
df_filtrado.to_excel(nome_arquivo_saida, index=False)

print(f"\n✅ Análise concluída! O arquivo '{nome_arquivo_saida}' foi salvo na sua pasta.")
print("Ele contém todas as descrições originais lado a lado com a indicação (0 ou 1) de qual grupo semântico cada documento pertence.")

In [ ]:
############################################
# SEPARAÇÃO SEMÂNTICA EM LOTE
############################################

import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

# 1. Configurar pasta de saída para as imagens
pasta_graficos = "graficos_auditoria"
os.makedirs(pasta_graficos, exist_ok=True)

# 2. Carregar a base de dados

# Mapeamento padronizado das colunas do arquivo Excel

# Coluna A (SN) = identificador único do documento
# Coluna F (BOX) = número de identificação do box físico
# Coluna I (COD) = código de classificação conforme a Tabela de Temporalidade
# Coluna O (DESC) = Descrição do documento

# O Python começa no índice zero
colunas_alvo = [0, 5, 8, 14]
df = pd.read_excel('/content/dados_AP.xlsx', usecols=colunas_alvo, names=['SN', 'BOX','COD', 'DESC'])


# Limpar linhas inválidas
df = df.dropna(subset=['COD', 'DESC'])
df['DESC'] = df['DESC'].astype(str)

# 3. Identificar todos os códigos de temporalidade únicos
codigos_unicos = df['COD'].unique()
print(f"Total de códigos únicos para auditar: {len(codigos_unicos)}\n")

stop_words_pt = ['de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'uma', 'para', 'com', 'os', 'as', 'na', 'no', 'dos', 'das', 'sem', 'ao', 'aos']
resultados_auditoria = []

# 4. Iniciar a Varredura
for codigo in codigos_unicos:
    df_filtrado = df[df['COD'] == codigo].copy()

    # Ignorar amostras muito pequenas que não justificam análise estatística
    if len(df_filtrado) < 15:
        resultados_auditoria.append({'Código': codigo, 'Status': 'Amostra Pequena', 'Score': 0, 'Volume de Documentos': len(df_filtrado)})
        continue

    vectorizer = TfidfVectorizer(stop_words=stop_words_pt)

    try:
        X = vectorizer.fit_transform(df_filtrado['DESC'])

        # NOVA ETAPA: Reduzir a dimensionalidade ANTES de classificar
        # Comprime de centenas de palavras para os 10 "temas" principais
        svd = TruncatedSVD(n_components=min(10, X.shape[1]-1), random_state=42)
        X_reduzido = svd.fit_transform(X)

        # O algoritmo agora trabalha nos dados "limpos" de ruído
        kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
        df_filtrado['Novo_Grupo'] = kmeans.fit_predict(X_reduzido)

        # O score agora refletirá a separação real (deve subir substancialmente)
        score = silhouette_score(X_reduzido, kmeans.labels_)

        resultados_auditoria.append({
            'Código': codigo,
            'Status': 'Analisado',
            'Score': round(score, 3),
            'Volume de Documentos': len(df_filtrado)
        })

        # 5. Salvar o gráfico APENAS se o score justificar a revisão (Score > 0.25)
        if score >= 0.25:
           # Gráfico usa apenas as 2 primeiras dimensões para visualizar na tela
            coordenadas_2d = X_reduzido[:, :2]

            plt.figure(figsize=(9, 6))
            plt.scatter(coordenadas_2d[:, 0], coordenadas_2d[:, 1],
                        c=df_filtrado['Novo_Grupo'], cmap='Blues',
                        edgecolor='navy', alpha=0.7, s=80)

            plt.title(f"Revisão Recomendada - Código: {codigo}\nScore de Silhueta: {score:.3f}", fontsize=14, color='navy')
            plt.xlabel("Componente Principal 1")
            plt.ylabel("Componente Principal 2")
            plt.grid(True, linestyle='--', alpha=0.5)

            # Salvar imagem e fechar para liberar memória do sistema
            nome_arquivo = f"{pasta_graficos}/revisao_{str(codigo).replace('.', '_')}.png"
            plt.tight_layout()
            plt.savefig(nome_arquivo, dpi=150)
            plt.close()

    except ValueError:
        # Captura códigos cujos textos não geraram vocabulário útil
        resultados_auditoria.append({'Código': codigo, 'Status': 'Vocabulário Vazio/Genérico', 'Score': 0, 'Volume de Documentos': len(df_filtrado)})

# 6. Gerar e Exportar o Relatório Final
df_relatorio = pd.DataFrame(resultados_auditoria)

# Ordenar para que os piores códigos (maior score = mais divididos) apareçam no topo
df_relatorio = df_relatorio.sort_values(by='Score', ascending=False)
df_relatorio.to_excel("relatorio_auditoria_completa.xlsx", index=False)

print("✅ Varredura finalizada!")
print(f"Os gráficos dos códigos problemáticos foram salvos na pasta: '{pasta_graficos}'")
print("O ranking completo foi salvo no arquivo: 'relatorio_auditoria_completa.xlsx'")

# Imprimir uma prévia no terminal para os códigos mais urgentes
urgentes = df_relatorio[df_relatorio['Score'] >= 0.25]
if not urgentes.empty:
    print("\n⚠️  CÓDIGOS QUE EXIGEM REVISÃO URGENTE:")
    print(urgentes[['Código', 'Score', 'Volume de Documentos']].head(10).to_string(index=False))